In [8]:
import pandas as pd
import pyodbc

# Load CSV
df = pd.read_csv(r"C:\Users\Anom\Downloads\archive\DataCoSupplyChainDataset.csv", encoding='latin-1')

# Connect to SQL Server
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=DataCoSupplyChain;"
    "Trusted_Connection=yes;"
)
cursor = conn.cursor()

# Drop table if exists from failed wizard attempt
cursor.execute("IF OBJECT_ID('supply_chain_raw', 'U') IS NOT NULL DROP TABLE supply_chain_raw")
conn.commit()

# Clean column names for SQL (remove spaces, brackets, slashes)
df.columns = (df.columns
              .str.strip()
              .str.replace(' ', '_')
              .str.replace('(', '')
              .str.replace(')', '')
              .str.replace('/', '_'))

print("Cleaned columns:")
print(df.columns.tolist())
print(f"\nShape: {df.shape}")

Cleaned columns:
['Type', 'Days_for_shipping_real', 'Days_for_shipment_scheduled', 'Benefit_per_order', 'Sales_per_customer', 'Delivery_Status', 'Late_delivery_risk', 'Category_Id', 'Category_Name', 'Customer_City', 'Customer_Country', 'Customer_Email', 'Customer_Fname', 'Customer_Id', 'Customer_Lname', 'Customer_Password', 'Customer_Segment', 'Customer_State', 'Customer_Street', 'Customer_Zipcode', 'Department_Id', 'Department_Name', 'Latitude', 'Longitude', 'Market', 'Order_City', 'Order_Country', 'Order_Customer_Id', 'order_date_DateOrders', 'Order_Id', 'Order_Item_Cardprod_Id', 'Order_Item_Discount', 'Order_Item_Discount_Rate', 'Order_Item_Id', 'Order_Item_Product_Price', 'Order_Item_Profit_Ratio', 'Order_Item_Quantity', 'Sales', 'Order_Item_Total', 'Order_Profit_Per_Order', 'Order_Region', 'Order_State', 'Order_Status', 'Order_Zipcode', 'Product_Card_Id', 'Product_Category_Id', 'Product_Description', 'Product_Image', 'Product_Name', 'Product_Price', 'Product_Status', 'shipping_dat

In [3]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'PostgreSQL ANSI(x64)', 'PostgreSQL Unicode(x64)', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server']


In [9]:
import pandas as pd
import pyodbc

df = pd.read_csv(r"C:\Users\Anom\Downloads\archive\DataCoSupplyChainDataset.csv", encoding='latin-1')

# Clean column names
df.columns = (df.columns
              .str.strip()
              .str.replace(' ', '_')
              .str.replace('(', '')
              .str.replace(')', '')
              .str.replace('/', '_'))

# Connect using Driver 18
conn = pyodbc.connect(
    "DRIVER={ODBC Driver 18 for SQL Server};"
    "SERVER=localhost\\SQLEXPRESS;"
    "DATABASE=DataCoSupplyChain;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)
cursor = conn.cursor()
print("Connected successfully!")

# Drop table if exists from failed wizard attempt
cursor.execute("IF OBJECT_ID('supply_chain_raw', 'U') IS NOT NULL DROP TABLE supply_chain_raw")
conn.commit()
print("Cleaned up old table.")

Connected successfully!
Cleaned up old table.


In [11]:
# Drop and recreate table with safer types
cursor.execute("IF OBJECT_ID('supply_chain_raw', 'U') IS NOT NULL DROP TABLE supply_chain_raw")
conn.commit()

cursor.execute("""
CREATE TABLE supply_chain_raw (
    Type NVARCHAR(50),
    Days_for_shipping_real FLOAT,
    Days_for_shipment_scheduled FLOAT,
    Benefit_per_order FLOAT,
    Sales_per_customer FLOAT,
    Delivery_Status NVARCHAR(50),
    Late_delivery_risk FLOAT,
    Category_Id FLOAT,
    Category_Name NVARCHAR(100),
    Customer_City NVARCHAR(100),
    Customer_Country NVARCHAR(100),
    Customer_Email NVARCHAR(100),
    Customer_Fname NVARCHAR(50),
    Customer_Id FLOAT,
    Customer_Lname NVARCHAR(50),
    Customer_Password NVARCHAR(100),
    Customer_Segment NVARCHAR(50),
    Customer_State NVARCHAR(100),
    Customer_Street NVARCHAR(200),
    Customer_Zipcode NVARCHAR(20),
    Department_Id FLOAT,
    Department_Name NVARCHAR(100),
    Latitude FLOAT,
    Longitude FLOAT,
    Market NVARCHAR(50),
    Order_City NVARCHAR(100),
    Order_Country NVARCHAR(100),
    Order_Customer_Id FLOAT,
    order_date_DateOrders NVARCHAR(50),
    Order_Id FLOAT,
    Order_Item_Cardprod_Id FLOAT,
    Order_Item_Discount FLOAT,
    Order_Item_Discount_Rate FLOAT,
    Order_Item_Id FLOAT,
    Order_Item_Product_Price FLOAT,
    Order_Item_Profit_Ratio FLOAT,
    Order_Item_Quantity FLOAT,
    Sales FLOAT,
    Order_Item_Total FLOAT,
    Order_Profit_Per_Order FLOAT,
    Order_Region NVARCHAR(100),
    Order_State NVARCHAR(100),
    Order_Status NVARCHAR(50),
    Order_Zipcode NVARCHAR(20),
    Product_Card_Id FLOAT,
    Product_Category_Id FLOAT,
    Product_Description NVARCHAR(MAX),
    Product_Image NVARCHAR(500),
    Product_Name NVARCHAR(200),
    Product_Price FLOAT,
    Product_Status FLOAT,
    shipping_date_DateOrders NVARCHAR(50),
    Shipping_Mode NVARCHAR(50)
)
""")
conn.commit()
print("Table recreated with safe types.")

# Insert again
cursor.fast_executemany = True
cols = ', '.join(df.columns.tolist())
placeholders = ', '.join(['?'] * len(df.columns))
sql = f"INSERT INTO supply_chain_raw ({cols}) VALUES ({placeholders})"

import numpy as np
df = df.where(pd.notnull(df), None)

chunk_size = 5000
total = len(df)
for i in range(0, total, chunk_size):
    chunk = df.iloc[i:i+chunk_size]
    cursor.executemany(sql, chunk.values.tolist())
    conn.commit()
    print(f"Inserted {min(i+chunk_size, total)}/{total} rows...")

print("All data loaded successfully!")

Table recreated with safe types.


DataError: ('22003', '[22003] [Microsoft][ODBC Driver 18 for SQL Server]Numeric value out of range (0) (SQLExecute)')

In [12]:
# Find which column has the problematic value
import numpy as np

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for col in numeric_cols:
    try:
        max_val = df[col].max()
        min_val = df[col].min()
        if max_val > 1e15 or min_val < -1e15:
            print(f"PROBLEM: {col} | min: {min_val} | max: {max_val}")
        else:
            print(f"OK: {col} | min: {min_val} | max: {max_val}")
    except Exception as e:
        print(f"ERROR in {col}: {e}")

OK: Days_for_shipping_real | min: 0 | max: 6
OK: Days_for_shipment_scheduled | min: 0 | max: 4
OK: Benefit_per_order | min: -4274.97998 | max: 911.7999878
OK: Sales_per_customer | min: 7.489999771 | max: 1939.98999
OK: Late_delivery_risk | min: 0 | max: 1
OK: Category_Id | min: 2 | max: 76
OK: Customer_Id | min: 1 | max: 20757
OK: Customer_Zipcode | min: 603.0 | max: 99205.0
OK: Department_Id | min: 2 | max: 12
OK: Latitude | min: -33.93755341 | max: 48.78193283
OK: Longitude | min: -158.0259857 | max: 115.2630768
OK: Order_Customer_Id | min: 1 | max: 20757
OK: Order_Id | min: 1 | max: 77204
OK: Order_Item_Cardprod_Id | min: 19 | max: 1363
OK: Order_Item_Discount | min: 0.0 | max: 500.0
OK: Order_Item_Discount_Rate | min: 0.0 | max: 0.25
OK: Order_Item_Id | min: 1 | max: 180519
OK: Order_Item_Product_Price | min: 9.989999771 | max: 1999.98999
OK: Order_Item_Profit_Ratio | min: -2.75 | max: 0.5
OK: Order_Item_Quantity | min: 1 | max: 5
OK: Sales | min: 9.989999771 | max: 1999.98999
OK: 

In [13]:
# Install sqlalchemy if needed
# pip install sqlalchemy

from sqlalchemy import create_engine
import pandas as pd

# Reload fresh
df = pd.read_csv(r"C:\Users\Anom\Downloads\archive\DataCoSupplyChainDataset.csv", encoding='latin-1')

# Clean column names
df.columns = (df.columns
              .str.strip()
              .str.replace(' ', '_')
              .str.replace('(', '')
              .str.replace(')', '')
              .str.replace('/', '_'))

# Create engine
engine = create_engine(
    "mssql+pyodbc://localhost\\SQLEXPRESS/DataCoSupplyChain"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&TrustServerCertificate=yes"
)

# Load in chunks with progress
chunk_size = 5000
total = len(df)
for i in range(0, total, chunk_size):
    chunk = df.iloc[i:i+chunk_size]
    if_exists = 'replace' if i == 0 else 'append'
    chunk.to_sql('supply_chain_raw', engine, if_exists=if_exists, index=False)
    print(f"Inserted {min(i+chunk_size, total)}/{total} rows...")

print("Done!")

D:\anaconda\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Inserted 5000/180519 rows...
Inserted 10000/180519 rows...
Inserted 15000/180519 rows...
Inserted 20000/180519 rows...
Inserted 25000/180519 rows...
Inserted 30000/180519 rows...
Inserted 35000/180519 rows...
Inserted 40000/180519 rows...
Inserted 45000/180519 rows...
Inserted 50000/180519 rows...
Inserted 55000/180519 rows...
Inserted 60000/180519 rows...
Inserted 65000/180519 rows...
Inserted 70000/180519 rows...
Inserted 75000/180519 rows...
Inserted 80000/180519 rows...
Inserted 85000/180519 rows...
Inserted 90000/180519 rows...
Inserted 95000/180519 rows...
Inserted 100000/180519 rows...
Inserted 105000/180519 rows...
Inserted 110000/180519 rows...
Inserted 115000/180519 rows...
Inserted 120000/180519 rows...
Inserted 125000/180519 rows...
Inserted 130000/180519 rows...
Inserted 135000/180519 rows...
Inserted 140000/180519 rows...
Inserted 145000/180519 rows...
Inserted 150000/180519 rows...
Inserted 155000/180519 rows...
Inserted 160000/180519 rows...
Inserted 165000/180519 rows.

In [15]:
pip install sqlalchemy pyodbc --upgrade

  Using cached pyodbc-5.3.0-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
Using cached pyodbc-5.3.0-cp312-cp312-win_amd64.whl (70 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'D:\\anaconda\\Lib\\site-packages\\pyodbc.cp312-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



In [18]:
from sqlalchemy import create_engine
import pandas as pd

df = pd.read_csv(r"C:\Users\Anom\Downloads\archive\DataCoSupplyChainDataset.csv", encoding='latin-1')

df.columns = (df.columns
              .str.strip()
              .str.replace(' ', '_')
              .str.replace('(', '')
              .str.replace(')', '')
              .str.replace('/', '_'))

engine = create_engine(
    "mssql+pyodbc://localhost\\SQLEXPRESS/DataCoSupplyChain"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&TrustServerCertificate=yes"
)

# Test connection
with engine.connect() as conn:
    print("Connection successful!")

# Load data
chunk_size = 5000
total = len(df)
for i in range(0, total, chunk_size):
    chunk = df.iloc[i:i+chunk_size]
    if_exists = 'replace' if i == 0 else 'append'
    chunk.to_sql('supply_chain_raw', engine, if_exists=if_exists, index=False)
    print(f"Inserted {min(i+chunk_size, total)}/{total} rows...")

print("Done!")

C:\Users\Anom\AppData\Local\Temp\ipykernel_25248\2699109731.py:21: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Connection successful!
Inserted 5000/180519 rows...
Inserted 10000/180519 rows...
Inserted 15000/180519 rows...
Inserted 20000/180519 rows...
Inserted 25000/180519 rows...
Inserted 30000/180519 rows...
Inserted 35000/180519 rows...
Inserted 40000/180519 rows...
Inserted 45000/180519 rows...
Inserted 50000/180519 rows...
Inserted 55000/180519 rows...
Inserted 60000/180519 rows...
Inserted 65000/180519 rows...
Inserted 70000/180519 rows...
Inserted 75000/180519 rows...
Inserted 80000/180519 rows...
Inserted 85000/180519 rows...
Inserted 90000/180519 rows...
Inserted 95000/180519 rows...
Inserted 100000/180519 rows...
Inserted 105000/180519 rows...
Inserted 110000/180519 rows...
Inserted 115000/180519 rows...
Inserted 120000/180519 rows...
Inserted 125000/180519 rows...
Inserted 130000/180519 rows...
Inserted 135000/180519 rows...
Inserted 140000/180519 rows...
Inserted 145000/180519 rows...
Inserted 150000/180519 rows...
Inserted 155000/180519 rows...
Inserted 160000/180519 rows...
Inser

In [19]:
with engine.connect() as conn:
    result = conn.execute(__import__('sqlalchemy').text("SELECT COUNT(*) FROM supply_chain_raw"))
    print(f"Rows in SQL Server: {result.fetchone()[0]}")

Rows in SQL Server: 180519
